# Objects, names, and the data model

This is one of the most important conceptual modules in the entire course because it replaces a weak beginner mental model with a precise one. In Python, a variable is not a box holding a value. A **name** is a label bound to an **object**. Multiple names can refer to the same object, and whether changes are shared depends on whether the object is mutable.

That single idea explains aliasing bugs, mutable default argument surprises, confusion around `is` versus `==`, and many copy-related mistakes. Once you see names and objects separately, the behaviour becomes easier to predict.

As you work through the notebook, distinguish carefully between **rebinding a name** and **mutating an object**. Similar syntax can produce very different results.

## Visual model

```text
a -----> [1, 2, 3] <----- b

rebind a:   a -----> [9, 9]
mutate b:   a and b still point to the SAME object
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 4. `is` versus `==`

In [ ]:
a is b      # identity: are these the SAME object?         (id(a) == id(b))
a == b      # equality: do these objects COMPARE equal?    (calls __eq__)

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
a == b     # True   -- same contents
a is b     # False  -- two distinct objects

**The rule: use `is` only for singletons.**

In [ ]:
if x is None: ...          # correct, always
if x is True: ...          # correct but usually unnecessary; prefer `if x:`
if flag is Sentinel: ...   # correct for your own sentinel objects
if name is "admin": ...    # WRONG. Use ==. It may appear to work. It is a bug.

Why is `is` wrong for values? Because whether two equal values are the same
object is an implementation detail:

```text
>>> a = 256; b = 256; a is b
True
>>> a = 257; b = 257; a is b
False          # in a REPL. In a single compiled block, possibly True.
```


CPython pre-allocates the integers -5 through 256 at startup and reuses them.
That is **small-int caching**, a CPython optimisation, not a language rule.
String literals get similar treatment (**interning**) for identifier-like
strings. Python 3.8+ emits a `SyntaxWarning` for `is` with a literal, precisely
because this bug was so common.

```text
>>> x = "hello"; y = "hello"; x is y
True                    # both interned
>>> x = "hello world!"; y = "hello world!"; x is y
False                   # not interned (contains characters that make it
                        #  ineligible under the current heuristic)
```


Never build logic on any of this. Use `==` for values, `is` for `None` and
sentinels. Full stop.

### The sentinel pattern

`is` has one genuinely important use beyond `None`: distinguishing "not
provided" from "provided as None".

In [ ]:
_MISSING = object()      # a unique object that equals nothing else

def get(config: dict[str, object], key: str, default: object = _MISSING) -> object:
    if key in config:
        return config[key]
    if default is _MISSING:
        raise KeyError(key)      # caller gave no default: missing is an error
    return default               # caller gave a default, possibly None

You will see this in the standard library and in every serious library. It is
the only way to let `None` be a legitimate default value.

---

## Concept 5. Function arguments: call by object reference

Python is neither "pass by value" nor "pass by reference". Both terms mislead,
and material using them is a reliable signal that the author has not thought
about it. Python passes **object references, by value**. The parameter name is
a new name bound to the same object.

In [ ]:
def rebind(lst: list[int]) -> None:
    lst = [9, 9, 9]        # rebinds the LOCAL name. Caller sees nothing.

def mutate(lst: list[int]) -> None:
    lst.append(9)          # mutates the SHARED object. Caller sees it.

data = [1, 2]
rebind(data);  print(data)      # [1, 2]
mutate(data);  print(data)      # [1, 2, 9]

Same parameter, same call syntax, opposite effect — determined entirely by
whether the body rebinds or mutates.

### The mutable default argument

The most famous Python bug, and now you can explain it rather than memorise it.

In [ ]:
def add_item(item: str, basket: list[str] = []) -> list[str]:
    basket.append(item)
    return basket

print(add_item("apple"))    # ['apple']
print(add_item("pear"))     # ['apple', 'pear']    <-- !

**Default arguments are evaluated once, when the `def` statement executes**, not
on each call. That one list object is stored on the function and reused forever.
You can see it:

```text
>>> add_item.__defaults__
(['apple', 'pear'],)
```


The fix, every time:

In [ ]:
def add_item(item: str, basket: list[str] | None = None) -> list[str]:
    if basket is None:
        basket = []
    basket.append(item)
    return basket

Note `is None`, not `== None` or `if not basket` — an empty list passed
deliberately is falsy and would be silently replaced.

The same trap applies to `{}`, `set()`, and to any expression evaluated at def
time: `def log(t=datetime.now())` freezes the timestamp at import.

`ruff` catches this with rule `B006`, which is enabled in this course's config.

---

## Concept 6. Copying

In [ ]:
import copy

original = [[1, 2], [3, 4]]

alias    = original                 # same object
shallow  = original[:]              # new outer list, SAME inner lists
shallow2 = list(original)           # identical to the above
shallow3 = copy.copy(original)      # identical to the above
deep     = copy.deepcopy(original)  # new outer AND new inner objects

original[0].append(99)
print(alias)     # [[1, 2, 99], [3, 4]]
print(shallow)   # [[1, 2, 99], [3, 4]]   <-- shared inner list
print(deep)      # [[1, 2], [3, 4]]       <-- fully independent

```text
alias    ────────────────> [ ● , ● ]
                             │   │
original ──────────────────> │   │
                             v   v
shallow  ──> [ ● , ● ] ────> [1,2] [3,4]
                ^ ^            ^     ^
                └─┴────────────┴─────┘   (shared!)

deep     ──> [ ● , ● ] ────> [1,2]' [3,4]'   (fresh copies)
```


Practical guidance:

- A shallow copy is enough when the contents are immutable. `list(nums)` for a
  list of ints is genuinely safe.
- `deepcopy` is correct but slow, and it recurses through everything reachable —
  including, by accident, a database connection or a whole object graph.
- The best answer is usually **avoid needing a copy**: use immutable data, or
  return new objects instead of mutating in place. This is the theme that
  Modules 11 and 14 build on.

---

## Concept 7. Memory: reference counting and the cycle collector

CPython frees an object when its reference count hits zero.

In [ ]:
import sys

a = [1, 2, 3]
sys.getrefcount(a)      # 2: one for `a`, one for the temporary argument
b = a
sys.getrefcount(a)      # 3
del b
sys.getrefcount(a)      # 2

Refcounting is immediate and predictable, which is why this works:

In [ ]:
with open("f.txt") as fh:      # closed deterministically at the end of the block
    ...

But refcounting alone cannot free a **cycle**:

In [ ]:
a = {}; b = {}
a["b"] = b; b["a"] = a         # each holds a reference to the other
del a, b                        # refcounts are still 1 each. Unreachable, but not freed.

That is why CPython also runs a **generational cycle collector** (`gc` module).
It finds unreachable cycles periodically. Two consequences worth carrying:

1. Object destruction is *usually* immediate but not *guaranteed* immediate.
   Never rely on `__del__` for cleanup — use a context manager (Module 09).
2. Cycles are collected, so they are not a leak, but they delay collection and
   cost CPU. `weakref` breaks them where it matters (caches, parent pointers,
   observer registries).

Other implementations (PyPy, GraalPy) do not refcount at all. Code that assumes
"the file closes when the variable goes out of scope" breaks on them. Use `with`.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Everything is an object
- Section 2: Names are not boxes
- Section 3: Mutable and immutable
- Section 4: `is` versus `==`
- Section 5: Function arguments: call by object reference
- Section 6: Copying
- Section 7: Memory: reference counting and the cycle collector
- Section 8: Truthiness
- Section 9: Namespaces are dictionaries

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import copy

---

## `q01`

_q01_

In [ ]:
def q01() -> None:
    # PREDICTION:
    a = [1, 2, 3]
    b = a
    b.append(4)
    print("q01", a)

---

## `q02`

_q02_

In [ ]:
def q02() -> None:
    # PREDICTION:
    a = [1, 2, 3]
    b = a
    b = [9, 9, 9]
    print("q02", a)

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION:
    a = [1, 2]
    b = a
    b += [3]
    print("q03", a, a is b)

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION:
    a = [1, 2]
    b = a
    b = b + [3]
    print("q04", a, a is b)

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION:
    t = (1, 2)
    u = t
    u += (3,)
    print("q05", t, u, t is u)

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION:
    grid = [[0] * 3] * 3
    grid[0][0] = 1
    print("q06", grid)

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # PREDICTION:
    def add(item: str, bag: list[str] = []) -> list[str]:
        bag.append(item)
        return bag

    print("q07", add("a"), add("b"), add("c"))

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION:
    outer = [[1, 2], [3, 4]]
    shallow = copy.copy(outer)
    deep = copy.deepcopy(outer)
    outer[0].append(99)
    print("q08", shallow, deep)

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION: (and explain WHY, not just what)
    a, b = 256, 256
    c, d = 1000, 1000
    print("q09", a is b, c is d)

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION:
    t = ([1, 2], "x")
    t[0].append(3)
    print("q10", t)
    try:
        t[0] = [9]
    except Exception as exc:
        print("q10 error:", type(exc).__name__)

---

## `q11`

_q11_

In [ ]:
def q11() -> None:
    # PREDICTION:
    nums = [1, 2, 3, 4, 5, 6]
    for n in nums:
        if n % 2 == 0:
            nums.remove(n)
    print("q11", nums)

---

## `q12`

_q12_

In [ ]:
def q12() -> None:
    # PREDICTION:
    a = [3, 1, 2]
    b = a.sort()
    c = sorted([3, 1, 2])
    print("q12", a, b, c)

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        fn()

    print(
        "\nNow write, for each question you got wrong, ONE sentence naming the\n"
        "mechanism you misjudged. Not 'I forgot' -- the actual mechanism.\n"
        "Put those sentences in PROGRESS.md's mistakes log."
    )

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.